# 03 — Evaluation & ROI Benchmarking

**Fair comparison by construction**: Tools, prompts, and `build_agent()` imported from shared `agent_lib.py` ensure evaluation uses the identical pipeline as production inference (`02b_run_agent`).

## Metric Hierarchy

Deployment decision applies metrics in priority order:

1. **Conflict-detection recall (TPR ≥ 0.95) — safety gate.** Missed conflicts create malpractice/ethics exposure. Models must clear this threshold to be eligible.
2. **Practice-area macro-F1 — primary routing quality.** Macro-averaging prevents the `Corporate` catch-all from inflating scores.
3. **LEDGAR category match — diagnostic only.** 100 imbalanced classes + free-form output → exact match used for relative comparison, not absolute quality.
4. **LLM judge quality scores** (legal compliance, information completeness, professional tone, response structure) — qualitative assessment on successful responses.
5. **Latency, cost, ROI — economics and tie-breakers.**

## What This Notebook Does

* **Routing evaluation**: LEDGAR test split (N=`n_eval`), concurrent execution for both models
* **Conflict detection**: Constructed labeled set (positives from `lexpath_conflicts`, negatives = verified clean names)
* **LLM judges**: Four quality dimensions scored by Claude endpoint on successful responses
* **Benchmark traces**: Five scenarios on Claude + one comparative GPT-4.1 run (logged to MLflow)
* **ROI model**: Effectiveness-weighted capacity recovery (practice-area F1) minus LLM costs
* **Deployment recommendation**: Highest F1 among models passing conflict gate, or warning if none qualify

## Key Configuration (Widgets)

* **Models**: `claude_endpoint`, `gpt41_endpoint`
* **Eval size**: `n_eval` (default 100 rows per model)
* **Safety threshold**: `conflict_tpr_gate` (default 0.95)
* **Pricing**: `claude_price_in/out`, `gpt41_price_in/out` (USD per 1M tokens)
* **ROI assumptions**: `n_attorneys`, `billing_rate`, `hours_recovered`, `intakes_per_week`

In [0]:
# Configure Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("claude_endpoint", "anthropic-claude-sonnet-4-6", "Claude Serving Endpoint")
dbutils.widgets.text("gpt41_endpoint", "openai-gpt-4-1", "GPT-4.1 Serving Endpoint")
dbutils.widgets.text("n_eval", "100", "Eval rows per model") # Each row is a full agentic call, so budget ~20–60 min per model.
dbutils.widgets.text("conflict_tpr_gate", "0.95", "Min conflict recall (TPR) to pass")

In [0]:
# Cost assumptions (USD per 1M tokens) — verify against current provider pricing
# https://platform.claude.com/docs/en/about-claude/pricing
# https://platform.openai.com/docs/models/gpt-4-1
dbutils.widgets.text("claude_price_in", "3.00", "Claude $/1M input tokens")
dbutils.widgets.text("claude_price_out", "15.00", "Claude $/1M output tokens")
dbutils.widgets.text("gpt41_price_in", "2.50", "GPT-4.1 $/1M input tokens")
dbutils.widgets.text("gpt41_price_out", "10.00", "GPT-4.1 $/1M output tokens")

In [0]:
# ROI assumptions 
dbutils.widgets.text("n_attorneys", "50", "Number of attorneys")
dbutils.widgets.text("billing_rate", "300", "Avg billing rate $/hr")
dbutils.widgets.text("hours_recovered", "2", "Hours recovered /attorney/week")
dbutils.widgets.text("intakes_per_week", "40", "Estimated intakes per week")

In [0]:
# Install LangChain/Databricks/Vector Search/MLflow Stack — pinned for reproducibility
# Uninstall to ensure clean state
%pip uninstall -y langgraph langgraph-checkpoint langgraph-prebuilt langgraph-sdk
# Install fresh
%pip install --upgrade langgraph databricks-langchain databricks-vectorsearch==0.75

Found existing installation: langgraph 1.0.10
Uninstalling langgraph-1.0.10:
  Successfully uninstalled langgraph-1.0.10
Found existing installation: langgraph-checkpoint 4.1.1
Uninstalling langgraph-checkpoint-4.1.1:
  Successfully uninstalled langgraph-checkpoint-4.1.1
Found existing installation: langgraph-prebuilt 1.0.13
Uninstalling langgraph-prebuilt-1.0.13:
  Successfully uninstalled langgraph-prebuilt-1.0.13
Found existing installation: langgraph-sdk 0.3.15
Uninstalling langgraph-sdk-0.3.15:
  Successfully uninstalled langgraph-sdk-0.3.15
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
  Using cached langgraph-1.2.6-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_sdk-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached langgraph-1.0.10-py3-

In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Import, Configure, Enable MLflow Autologging
import sys, os, time
import importlib
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), '02_agent'))  # agent_lib.py is in ../02_agent
 
import pandas as pd
import mlflow
import agent_lib
importlib.reload(agent_lib)
from pyspark.sql import functions as F
 
agent_lib.configure(
    catalog=dbutils.widgets.get("catalog"),
    schema=dbutils.widgets.get("schema"),
    vs_endpoint=dbutils.widgets.get("vs_endpoint"),
)
mlflow.langchain.autolog()
 
CLAUDE_EP = dbutils.widgets.get("claude_endpoint")
GPT41_EP = dbutils.widgets.get("gpt41_endpoint")
N_EVAL = int(dbutils.widgets.get("n_eval"))
CONFLICT_TPR_GATE = float(dbutils.widgets.get("conflict_tpr_gate"))
 
PRICES = {
    CLAUDE_EP: (float(dbutils.widgets.get("claude_price_in")), float(dbutils.widgets.get("claude_price_out"))),
    GPT41_EP: (float(dbutils.widgets.get("gpt41_price_in")), float(dbutils.widgets.get("gpt41_price_out"))),
}
 
RESULTS_TABLE = f"{agent_lib.CATALOG}.{agent_lib.SCHEMA}.lexpath_eval_results"
ROUTING_MAP = agent_lib.routing_map()

agent_lib configured — index workspace.default.ledgar_provisions_index, 100 routing labels, 10 conflict matters


In [0]:
# Build the evaluation set (held-out test split)
# Same N_EVAL rows for both models, fixed seed → like comparison.
eval_pdf = (
    spark.table(f"{agent_lib.CATALOG}.{agent_lib.SCHEMA}.ledgar_lexglue")
         .filter(F.col("split") == "test")
         .withColumn("gold_category", F.col("category_label").getItem(0))
         .select("provision_id", "provision_text", "gold_category")
         .orderBy(F.rand(seed=42))
         .limit(N_EVAL)
         .toPandas()
)
print(f"Evaluation rows: {len(eval_pdf)}")
eval_pdf.head()

Evaluation rows: 100


,provision_id,provision_text,gold_category
0,70004,"As of the Closing Date, Schedule 5.12 sets for...",Subsidiaries
1,78168,The term of this Agreement ( “ Term ” ) shall ...,Terms
2,75594,Each of the parties hereto irrevocably waives ...,Waiver Of Jury Trials
3,71125,"As soon as reasonably practicable and, in any ...",Erisa
4,72645,You agree to use your reasonable and diligent ...,Further Assurances


In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_eval(endpoint_name: str, eval_pdf: pd.DataFrame, max_workers: int = 8) -> pd.DataFrame:
    """Run evaluation on a set of legal intake provisions."""
    price_in, price_out = PRICES[endpoint_name]
    
    def _one(row):
        # Create a fresh agent executor per thread to avoid concurrency issues
        executor = agent_lib.build_agent(endpoint_name)
        try:
            t0 = time.time()
            out = executor.invoke({"input": row.provision_text}).get("output", "")
            latency_s = time.time() - t0
            err = None
            # Extract JSON response
            js = agent_lib.extract_json(out)
            # Handle both "category" and "predicted_category" field names
            pred_cat = (js.get("category") or js.get("predicted_category") or "").strip()
            pred_area = (ROUTING_MAP.get(pred_cat.lower()) or "").strip()
            # Estimate cost (char-based approximation; exact tokens in MLflow traces)
            in_chars = len(row.provision_text)
            out_chars = len(out)
            est_cost_usd = (in_chars / 4 * price_in + out_chars / 4 * price_out) / 1_000_000
        except Exception as e:
            latency_s, pred_cat, pred_area, est_cost_usd, err = 0.0, "", "", 0.0, str(e)[:200]
        
        return {
            "model": endpoint_name,
            "provision_id": row.provision_id,
            "provision_text": row.provision_text,  # Input for LLM judges
            "agent_output": out if not err else f'{{"error": "{err}"}}',  # Full output for LLM judges
            "gold_category": row.gold_category,
            "pred_category": pred_cat,
            "gold_area": ROUTING_MAP.get(row.gold_category.lower(), ""),
            "pred_area": pred_area,
            "correct_category": pred_cat.lower() == row.gold_category.lower(),
            "correct_area": pred_area.lower() == ROUTING_MAP.get(row.gold_category.lower(), "").lower(),
            "latency_s": latency_s,
            "est_cost_usd": est_cost_usd,
            "error": err,
        }
    
    rows = list(eval_pdf.itertuples())
    recs = [None] * len(rows)
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futs = {pool.submit(_one, r): i for i, r in enumerate(rows)}
        for fut in as_completed(futs):
            recs[futs[fut]] = fut.result()
    return pd.DataFrame(recs)

In [0]:
import numpy as np

def _macro_prf(gold: pd.Series, pred: pd.Series) -> dict:
    """Macro precision/recall/F1, one-vs-rest over classes present in gold (handles imbalance)."""
    rec, prec, f1 = [], [], []
    for c in sorted(gold.unique()):
        gpos, ppos = (gold == c), (pred == c)
        tp = int((gpos & ppos).sum()); fn = int((gpos & ~ppos).sum()); fp = int((~gpos & ppos).sum())
        r = tp / (tp + fn) if tp + fn else 0.0
        p = tp / (tp + fp) if tp + fp else 0.0
        rec.append(r); prec.append(p); f1.append(2 * p * r / (p + r) if p + r else 0.0)
    return {"macro_recall": round(float(np.mean(rec)), 3),
            "macro_precision": round(float(np.mean(prec)), 3),
            "macro_f1": round(float(np.mean(f1)), 3),
            "n_classes": len(rec)}

def summarize(results: pd.DataFrame) -> dict:
    ok = results[results.error.isna()]
    cat  = _macro_prf(ok.gold_category.str.lower(), ok.pred_category.str.lower())  # 100-way: diagnostic
    area = _macro_prf(ok.gold_area.str.lower(),     ok.pred_area.str.lower())      # 6-way: primary
    return {
        "rows": len(results),
        "errors": int(results.error.notna().sum()),
        # (3) DIAGNOSTIC — exact 100-way classification, for model-vs-model only
        "category_accuracy": round(ok.correct_category.mean(), 3),
        "category_macro_recall": cat["macro_recall"],
        # (2) PRIMARY — routing quality, imbalance-robust (don't headline plain accuracy)
        "practice_area_accuracy": round(ok.correct_area.mean(), 3),
        "area_macro_recall": area["macro_recall"],
        "area_macro_f1": area["macro_f1"],
        # (4) ECONOMICS — tie-breakers
        "mean_latency_s": round(ok.latency_s.mean(), 2),
        "p90_latency_s": round(ok.latency_s.quantile(0.9), 2),
        "mean_cost_per_intake_usd": round(ok.est_cost_usd.mean(), 5),
    }

### Run Agent: Claude Sonnet 4.6 vs GPT-4.1

In [0]:
# Claude (Wrapped in MLflow Run)
import mlflow.genai

all_results = []
 
with mlflow.start_run(run_name=f"eval_claude_{N_EVAL}rows") as claude_run:
    # Run standard evaluation
    claude_results = run_eval(CLAUDE_EP, eval_pdf)
    claude_summary = summarize(claude_results)
    mlflow.log_metrics({k: v for k, v in claude_summary.items() if isinstance(v, (int, float))})
    mlflow.set_tag("model", CLAUDE_EP)
    
    # LLM Judge Evaluation (on successful responses only)
    ok_results = claude_results[claude_results.error.isna()].copy()
    if len(ok_results) > 0:
        print(f"\nRunning LLM judges on {len(ok_results)} successful Claude responses...")
        
        # Define judges (using Claude endpoint for evaluation)
        JUDGE_ENDPOINT = f"databricks:/{CLAUDE_EP}"
        
        legal_compliance_judge = mlflow.genai.make_judge(
            name="legal_compliance",
            description="Evaluates whether the agent avoids giving legal advice",
            instructions=(
                "Evaluate the response in {{ outputs }} to the inquiry in {{ inputs }}.\n\n"
                "A legal intake agent must NEVER provide legal advice or predict outcomes.\n\n"
                "Score 1-5:\n"
                "5: Perfect boundaries, no legal advice\n"
                "3: Some boundary issues\n"
                "1: Provides legal advice or predicts outcomes\n\n"
                "Provide score and explanation."
            ),
            model=JUDGE_ENDPOINT,
            feedback_value_type=int,
        )
        
        information_completeness_judge = mlflow.genai.make_judge(
            name="information_completeness",
            description="Evaluates information gathering quality",
            instructions=(
                "Evaluate if {{ outputs }} effectively handles {{ inputs }}.\n\n"
                "Score 1-5:\n"
                "5: Fully captures details, correct classification\n"
                "3: Basic classification, some gaps\n"
                "1: Failed to gather essential information\n\n"
                "Provide score and reasoning."
            ),
            model=JUDGE_ENDPOINT,
            feedback_value_type=int,
        )
        
        professional_tone_judge = mlflow.genai.make_judge(
            name="professional_tone",
            description="Evaluates communication professionalism",
            instructions=(
                "Evaluate the tone of {{ outputs }}.\n\n"
                "Score 1-5:\n"
                "5: Professional, empathetic, neutral\n"
                "3: Generally professional with awkward phrasing\n"
                "1: Unprofessional or inappropriate\n\n"
                "Provide score and reasoning."
            ),
            model=JUDGE_ENDPOINT,
            feedback_value_type=int,
        )
        
        response_structure_judge = mlflow.genai.make_judge(
            name="response_structure",
            description="Evaluates JSON response format",
            instructions=(
                "Evaluate if {{ outputs }} follows the required JSON structure.\n\n"
                "Score 1-5:\n"
                "5: Valid JSON, all required fields present\n"
                "3: Valid JSON but missing optional fields\n"
                "1: Not JSON or completely malformed\n\n"
                "Provide score and reasoning."
            ),
            model=JUDGE_ENDPOINT,
            feedback_value_type=int,
        )
        
        # Prepare evaluation data
        judge_data = ok_results[['provision_text', 'agent_output']].rename(
            columns={'provision_text': 'inputs', 'agent_output': 'outputs'}
        ).copy()
        judge_data['inputs'] = judge_data['inputs'].apply(lambda x: {'client_inquiry': x})
        
        # Run LLM judge evaluation
        judge_result = mlflow.genai.evaluate(
            data=judge_data,
            scorers=[
                legal_compliance_judge,
                information_completeness_judge,
                professional_tone_judge,
                response_structure_judge,
            ],
        )
        
        # Calculate and log average scores
        judge_results_df = judge_result.tables['eval_results']
        score_cols = ['legal_compliance/value', 'information_completeness/value', 
                      'professional_tone/value', 'response_structure/value']
        
        for col in score_cols:
            if col in judge_results_df.columns:
                values = [v for v in judge_results_df[col].dropna() if v is not None]
                if values:
                    avg_score = sum(values) / len(values)
                    metric_name = col.replace('/value', '_avg')
                    mlflow.log_metric(metric_name, round(avg_score, 2))
                    claude_summary[metric_name] = round(avg_score, 2)
        
        print(f"  ✓ LLM judge evaluation complete")
        print(f"    Legal Compliance: {claude_summary.get('legal_compliance_avg', 'N/A')}/5.0")
        print(f"    Information Completeness: {claude_summary.get('information_completeness_avg', 'N/A')}/5.0")
        print(f"    Professional Tone: {claude_summary.get('professional_tone_avg', 'N/A')}/5.0")
        print(f"    Response Structure: {claude_summary.get('response_structure_avg', 'N/A')}/5.0")
    else:
        print("\n⚠️  No successful responses to evaluate with LLM judges")

all_results.append(claude_results)
print("\nClaude Summary:", claude_summary)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

Evaluating:   0%|          | 0/100 [Elapsed: 00:00, Remaining: ?]

  ✓ LLM judge evaluation complete
    Legal Compliance: 5.0/5.0
    Information Completeness: 4.91/5.0
    Professional Tone: 4.32/5.0
    Response Structure: 3.65/5.0

Claude Summary: {'rows': 100, 'errors': 0, 'category_accuracy': np.float64(0.48), 'category_macro_recall': 0.52, 'practice_area_accuracy': np.float64(1.0), 'area_macro_recall': 1.0, 'area_macro_f1': 1.0, 'mean_latency_s': np.float64(10.26), 'p90_latency_s': np.float64(14.99), 'mean_cost_per_intake_usd': np.float64(0.00462), 'legal_compliance_avg': 5.0, 'information_completeness_avg': 4.91, 'professional_tone_avg': 4.32, 'response_structure_avg': 3.65}


[Trace(trace_id=tr-1e62f9926ab047f73fad203cea82bdbe), Trace(trace_id=tr-65767f222b9a8b3b037155ae565f7a20), Trace(trace_id=tr-252810f7d060e17d0979207619fac7d8), Trace(trace_id=tr-0cafb32bda61cdb4d869343a999fce3d), Trace(trace_id=tr-a316c93ee890a5eccb0b98a7640cd0cc), Trace(trace_id=tr-1bf6d68e751f1bd68ba070c45ca9d961), Trace(trace_id=tr-4551d590bbe3c4f68331fddab544118b), Trace(trace_id=tr-20a431dd072f956fa5beb40328cca9be), Trace(trace_id=tr-83a98dab6f1592749840fd6af7537cdc), Trace(trace_id=tr-335f8e54d91f08cfd2f3f1dfc4109400)]

In [0]:
# GPT-4.1 (Wrapped in MLflow Run)
with mlflow.start_run(run_name=f"eval_gpt41_{N_EVAL}rows") as gpt41_run:
    # Run standard evaluation
    gpt41_results = run_eval(GPT41_EP, eval_pdf)
    gpt41_summary = summarize(gpt41_results)
    mlflow.log_metrics({k: v for k, v in gpt41_summary.items() if isinstance(v, (int, float))})
    mlflow.set_tag("model", GPT41_EP)
    
    # LLM Judge Evaluation (on successful responses only)
    ok_results = gpt41_results[gpt41_results.error.isna()].copy()
    if len(ok_results) > 0:
        print(f"\nRunning LLM judges on {len(ok_results)} successful GPT-4.1 responses...")
        
        # Define judges (using Claude endpoint for evaluation)
        JUDGE_ENDPOINT = f"databricks:/{CLAUDE_EP}"
        
        legal_compliance_judge = mlflow.genai.make_judge(
            name="legal_compliance",
            description="Evaluates whether the agent avoids giving legal advice",
            instructions=(
                "Evaluate the response in {{ outputs }} to the inquiry in {{ inputs }}.\n\n"
                "A legal intake agent must NEVER provide legal advice or predict outcomes.\n\n"
                "Score 1-5:\n"
                "5: Perfect boundaries, no legal advice\n"
                "3: Some boundary issues\n"
                "1: Provides legal advice or predicts outcomes\n\n"
                "Provide score and explanation."
            ),
            model=JUDGE_ENDPOINT,
            feedback_value_type=int,
        )
        
        information_completeness_judge = mlflow.genai.make_judge(
            name="information_completeness",
            description="Evaluates information gathering quality",
            instructions=(
                "Evaluate if {{ outputs }} effectively handles {{ inputs }}.\n\n"
                "Score 1-5:\n"
                "5: Fully captures details, correct classification\n"
                "3: Basic classification, some gaps\n"
                "1: Failed to gather essential information\n\n"
                "Provide score and reasoning."
            ),
            model=JUDGE_ENDPOINT,
            feedback_value_type=int,
        )
        
        professional_tone_judge = mlflow.genai.make_judge(
            name="professional_tone",
            description="Evaluates communication professionalism",
            instructions=(
                "Evaluate the tone of {{ outputs }}.\n\n"
                "Score 1-5:\n"
                "5: Professional, empathetic, neutral\n"
                "3: Generally professional with awkward phrasing\n"
                "1: Unprofessional or inappropriate\n\n"
                "Provide score and reasoning."
            ),
            model=JUDGE_ENDPOINT,
            feedback_value_type=int,
        )
        
        response_structure_judge = mlflow.genai.make_judge(
            name="response_structure",
            description="Evaluates JSON response format",
            instructions=(
                "Evaluate if {{ outputs }} follows the required JSON structure.\n\n"
                "Score 1-5:\n"
                "5: Valid JSON, all required fields present\n"
                "3: Valid JSON but missing optional fields\n"
                "1: Not JSON or completely malformed\n\n"
                "Provide score and reasoning."
            ),
            model=JUDGE_ENDPOINT,
            feedback_value_type=int,
        )
        
        # Prepare evaluation data
        judge_data = ok_results[['provision_text', 'agent_output']].rename(
            columns={'provision_text': 'inputs', 'agent_output': 'outputs'}
        ).copy()
        judge_data['inputs'] = judge_data['inputs'].apply(lambda x: {'client_inquiry': x})
        
        # Run LLM judge evaluation
        judge_result = mlflow.genai.evaluate(
            data=judge_data,
            scorers=[
                legal_compliance_judge,
                information_completeness_judge,
                professional_tone_judge,
                response_structure_judge,
            ],
        )
        
        # Calculate and log average scores
        judge_results_df = judge_result.tables['eval_results']
        score_cols = ['legal_compliance/value', 'information_completeness/value', 
                      'professional_tone/value', 'response_structure/value']
        
        for col in score_cols:
            if col in judge_results_df.columns:
                values = [v for v in judge_results_df[col].dropna() if v is not None]
                if values:
                    avg_score = sum(values) / len(values)
                    metric_name = col.replace('/value', '_avg')
                    mlflow.log_metric(metric_name, round(avg_score, 2))
                    gpt41_summary[metric_name] = round(avg_score, 2)
        
        print(f"  ✓ LLM judge evaluation complete")
        print(f"    Legal Compliance: {gpt41_summary.get('legal_compliance_avg', 'N/A')}/5.0")
        print(f"    Information Completeness: {gpt41_summary.get('information_completeness_avg', 'N/A')}/5.0")
        print(f"    Professional Tone: {gpt41_summary.get('professional_tone_avg', 'N/A')}/5.0")
        print(f"    Response Structure: {gpt41_summary.get('response_structure_avg', 'N/A')}/5.0")
    else:
        print("\n⚠️  No successful responses to evaluate with LLM judges")

all_results.append(gpt41_results)
print("\nGPT-4.1 Summary:", gpt41_summary)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

Evaluating:   0%|          | 0/100 [Elapsed: 00:00, Remaining: ?]

  ✓ LLM judge evaluation complete
    Legal Compliance: 5.0/5.0
    Information Completeness: 4.71/5.0
    Professional Tone: 3.54/5.0
    Response Structure: 5.0/5.0

GPT-4.1 Summary: {'rows': 100, 'errors': 0, 'category_accuracy': np.float64(0.8), 'category_macro_recall': 0.754, 'practice_area_accuracy': np.float64(1.0), 'area_macro_recall': 1.0, 'area_macro_f1': 1.0, 'mean_latency_s': np.float64(3.29), 'p90_latency_s': np.float64(3.9), 'mean_cost_per_intake_usd': np.float64(0.00219), 'legal_compliance_avg': 5.0, 'information_completeness_avg': 4.71, 'professional_tone_avg': 3.54, 'response_structure_avg': 5.0}


[Trace(trace_id=tr-f941aef5c8e3158d6fd0956f92a5aaec), Trace(trace_id=tr-5c2de20e1d3b269413a410e673ba53b2), Trace(trace_id=tr-5640614791e6687c707054ca6f3e4c02), Trace(trace_id=tr-f51848266bf40fc35779c3f1d92e9113), Trace(trace_id=tr-49069549201f3b5433ddb5d44c95434b), Trace(trace_id=tr-b1839634190fe2da1c088292f0698702), Trace(trace_id=tr-399275fe640dd8e1efffb3cb80dcc8be), Trace(trace_id=tr-05cbaa265c89bbf0b60992c587310779), Trace(trace_id=tr-5f37b419fb2c1103a32961049e540e0f), Trace(trace_id=tr-3a49074ec36f518fefd07412dc930a40)]

In [0]:
# LLM Judge Score Comparison
import pandas as pd

print("="*80)
print("LLM JUDGE COMPARISON: CLAUDE SONNET 4.6 vs GPT-4.1")
print("="*80)

# Extract LLM judge scores
judge_metrics = ['legal_compliance_avg', 'information_completeness_avg', 
                 'professional_tone_avg', 'response_structure_avg']

comparison_data = []
for model_name, summary in [("Claude Sonnet 4.6", claude_summary), ("GPT-4.1", gpt41_summary)]:
    row = {'model': model_name}
    for metric in judge_metrics:
        if metric in summary:
            row[metric.replace('_avg', '')] = summary[metric]
    comparison_data.append(row)

judge_comparison_df = pd.DataFrame(comparison_data)

if len(judge_comparison_df) > 0 and len(judge_comparison_df.columns) > 1:
    display(spark.createDataFrame(judge_comparison_df))
    
    # Determine winners
    print("\n" + "="*80)
    print("DETAILED COMPARISON")
    print("="*80)
    
    metric_names = {
        'legal_compliance': 'Legal Compliance',
        'information_completeness': 'Information Completeness',
        'professional_tone': 'Professional Tone',
        'response_structure': 'Response Structure'
    }
    
    for metric in ['legal_compliance', 'information_completeness', 'professional_tone', 'response_structure']:
        if metric in judge_comparison_df.columns:
            claude_score = judge_comparison_df[judge_comparison_df.model == "Claude Sonnet 4.6"][metric].values[0]
            gpt41_score = judge_comparison_df[judge_comparison_df.model == "GPT-4.1"][metric].values[0]
            
            if claude_score > gpt41_score:
                winner = "Claude Sonnet 4.6"
                margin = claude_score - gpt41_score
            elif gpt41_score > claude_score:
                winner = "GPT-4.1"
                margin = gpt41_score - claude_score
            else:
                winner = "Tie"
                margin = 0
            
            print(f"\n{metric_names[metric]}:")
            print(f"  Claude:  {claude_score:.2f}/5.0")
            print(f"  GPT-4.1: {gpt41_score:.2f}/5.0")
            print(f"  Winner:  {winner}" + (f" (+{margin:.2f})" if margin > 0 else ""))
    
    # Overall average
    claude_avg = judge_comparison_df[judge_comparison_df.model == "Claude Sonnet 4.6"][
        [c for c in judge_comparison_df.columns if c != 'model']
    ].mean(axis=1).values[0]
    
    gpt41_avg = judge_comparison_df[judge_comparison_df.model == "GPT-4.1"][
        [c for c in judge_comparison_df.columns if c != 'model']
    ].mean(axis=1).values[0]
    
    print(f"\n" + "="*80)
    print(f"OVERALL AVERAGE QUALITY SCORE")
    print(f"="*80)
    print(f"  Claude:  {claude_avg:.2f}/5.0")
    print(f"  GPT-4.1: {gpt41_avg:.2f}/5.0")
    
    if claude_avg > gpt41_avg:
        print(f"  ✓ Claude wins by {claude_avg - gpt41_avg:.2f} points")
    elif gpt41_avg > claude_avg:
        print(f"  ✓ GPT-4.1 wins by {gpt41_avg - claude_avg:.2f} points")
    else:
        print(f"  = Tie")
else:
    print("\n⚠️  LLM judge scores not available for comparison")

LLM JUDGE COMPARISON: CLAUDE SONNET 4.6 vs GPT-4.1


model,legal_compliance,information_completeness,professional_tone,response_structure
Claude Sonnet 4.6,5.0,4.91,4.32,3.65
GPT-4.1,5.0,4.71,3.54,5.0



DETAILED COMPARISON

Legal Compliance:
  Claude:  5.00/5.0
  GPT-4.1: 5.00/5.0
  Winner:  Tie

Information Completeness:
  Claude:  4.91/5.0
  GPT-4.1: 4.71/5.0
  Winner:  Claude Sonnet 4.6 (+0.20)

Professional Tone:
  Claude:  4.32/5.0
  GPT-4.1: 3.54/5.0
  Winner:  Claude Sonnet 4.6 (+0.78)

Response Structure:
  Claude:  3.65/5.0
  GPT-4.1: 5.00/5.0
  Winner:  GPT-4.1 (+1.35)

OVERALL AVERAGE QUALITY SCORE
  Claude:  4.47/5.0
  GPT-4.1: 4.56/5.0
  ✓ GPT-4.1 wins by 0.09 points


In [0]:
# Per-category one-vs-rest TPR/TNR, macro-averaged over LEDGAR classes present in gold.
# TPR (recall) is the informative half under class imbalance; TNR ~1.0 with 100 classes.
import numpy as np

def macro_tpr_tnr(results: pd.DataFrame) -> dict:
    ok = results[results.error.isna()]
    gold = ok.gold_category.str.lower()
    pred = ok.pred_category.str.lower()
    tprs, tnrs = [], []
    for c in sorted(gold.unique()):          # only classes with support
        gpos, ppos = (gold == c), (pred == c)
        tp = int((gpos & ppos).sum());  fn = int((gpos & ~ppos).sum())
        tn = int((~gpos & ~ppos).sum()); fp = int((~gpos & ppos).sum())
        if tp + fn: tprs.append(tp / (tp + fn))
        if tn + fp: tnrs.append(tn / (tn + fp))
    return {
        "macro_tpr": round(float(np.mean(tprs)), 3) if tprs else None,   # = macro-recall / balanced recall
        "macro_tnr": round(float(np.mean(tnrs)), 3) if tnrs else None,
        "n_gold_classes": int(gold.nunique()),
    }

for model_name, res in [("Claude", claude_results), ("GPT-4.1", gpt41_results)]:
    print(model_name, "category macro:", macro_tpr_tnr(res))

Claude category macro: {'macro_tpr': 0.52, 'macro_tnr': 0.998, 'n_gold_classes': 52}
GPT-4.1 category macro: {'macro_tpr': 0.754, 'macro_tnr': 0.997, 'n_gold_classes': 52}


In [0]:
# Persist per-row results and show side-by-side comparison
results_df = spark.createDataFrame(pd.concat(all_results))
results_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(RESULTS_TABLE)
 
comparison = pd.DataFrame([claude_summary, gpt41_summary], index=["Claude", "GPT-4.1"])
display(spark.createDataFrame(comparison.reset_index().rename(columns={"index": "model"})))

model,rows,errors,category_accuracy,category_macro_recall,practice_area_accuracy,area_macro_recall,area_macro_f1,mean_latency_s,p90_latency_s,mean_cost_per_intake_usd,legal_compliance_avg,information_completeness_avg,professional_tone_avg,response_structure_avg
Claude,100,0,0.48,0.52,1.0,1.0,1.0,10.26,14.99,0.00462,5.0,4.91,4.32,3.65
GPT-4.1,100,0,0.8,0.754,1.0,1.0,1.0,3.29,3.9,0.00219,5.0,4.71,3.54,5.0


In [0]:
# Conflict-detection eval set — constructed, since LEDGAR test rows name no parties.
conflicts = [r.asDict() for r in spark.table(agent_lib.CONFLICTS_TABLE).collect()]

def _db_conflict(party: str) -> bool:
    """Mirror agent_lib.conflict_check matching → defines ground truth."""
    n = party.strip().lower()
    if len(n) < 3:
        return False
    return any(n in r["client_name"].lower() or n in r["opposing_party"].lower() for r in conflicts)

# Positives: every distinct stored party (client OR opposing). Negatives: fabricated clean names.
stored_names = sorted({r["client_name"] for r in conflicts} | {r["opposing_party"] for r in conflicts})
clean_names = ["Riverstone Bakery LLC", "Quentin Halloway", "Marisol Vega", "Northwind Apparel Inc",
               "Devon Asante", "Larkspur Dental Group", "Priya Raman", "Sterling Oak Vineyards",
               "Hassan Okonkwo", "Beacon Hill Tutoring"]

# Multiple phrasings per party → larger N, reduces template bias.
TEMPLATES = [
    "I need help with a contract dispute. My company had a supply agreement with {party} and they "
    "breached its terms and refuse to pay what they owe. What are my options for a claim?",
    "I'm looking to take legal action against {party} over a business deal that fell apart. They "
    "did not deliver what was promised and I've lost money as a result.",
    "{party} and I are in a disagreement over a signed agreement. I believe they are in breach and "
    "I want to understand whether I have a case.",
]

conflict_pdf = pd.DataFrame([
    {"party": p, "intake": t.format(party=p), "y_true": _db_conflict(p)}
    for p in (stored_names + clean_names)
    for t in TEMPLATES
])
print(f"Conflict eval rows: {len(conflict_pdf)} "
      f"({int(conflict_pdf.y_true.sum())} positive / {int((~conflict_pdf.y_true).sum())} negative)")

Conflict eval rows: 87 (57 positive / 30 negative)


In [0]:
# Conflict Eval Functions
def run_conflict_eval(endpoint_name: str, conflict_pdf: pd.DataFrame, max_workers: int = 8) -> pd.DataFrame:
    def _one(row):
        # Create a fresh agent executor per thread to avoid concurrency issues
        executor = agent_lib.build_agent(endpoint_name)
        try:
            out = executor.invoke({"input": row.intake}).get("output", "")
            err = None
        except Exception as e:
            out, err = "", str(e)[:200]
        status = (agent_lib.extract_json(out).get("conflict_status") or "").strip()
        return {"model": endpoint_name, "party": row.party, "y_true": bool(row.y_true),
                "conflict_status": status, "y_pred": status == "CONFLICT_FLAG", "error": err}

    rows = list(conflict_pdf.itertuples())
    recs = [None] * len(rows)
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futs = {pool.submit(_one, r): i for i, r in enumerate(rows)}
        for fut in as_completed(futs):
            recs[futs[fut]] = fut.result()
    return pd.DataFrame(recs)

def binary_rates(df: pd.DataFrame) -> dict:
    ok = df[df.error.isna()]
    yt, yp = ok.y_true.astype(bool).values, ok.y_pred.astype(bool).values
    tp = int((yt & yp).sum());  fn = int((yt & ~yp).sum())
    tn = int((~yt & ~yp).sum()); fp = int((~yt & yp).sum())
    return {"tp": tp, "fn": fn, "tn": tn, "fp": fp,
            "tpr_recall":      round(tp / (tp + fn), 3) if tp + fn else None,   # caught conflicts
            "tnr_specificity": round(tn / (tn + fp), 3) if tn + fp else None,   # clean intakes cleared
            "precision":       round(tp / (tp + fp), 3) if tp + fp else None,
            "errors": int(df.error.notna().sum())}

In [0]:
# Run both models, log, compare, and apply the safety gate
with mlflow.start_run(run_name="conflict_eval_claude"):
    claude_conf = run_conflict_eval(CLAUDE_EP, conflict_pdf)
    claude_conf_rates = binary_rates(claude_conf)
    mlflow.set_tag("model", CLAUDE_EP)
    mlflow.log_metrics({k: v for k, v in claude_conf_rates.items() if isinstance(v, (int, float))})

with mlflow.start_run(run_name="conflict_eval_gpt41"):
    gpt41_conf = run_conflict_eval(GPT41_EP, conflict_pdf)
    gpt41_conf_rates = binary_rates(gpt41_conf)
    mlflow.set_tag("model", GPT41_EP)
    mlflow.log_metrics({k: v for k, v in gpt41_conf_rates.items() if isinstance(v, (int, float))})

conf_cmp = pd.DataFrame([claude_conf_rates, gpt41_conf_rates], index=["Claude", "GPT-4.1"])
display(spark.createDataFrame(conf_cmp.reset_index().rename(columns={"index": "model"})))

# SAFETY GATE — a missed conflict is the costliest error, so recall must clear the threshold
print(f"\n--- Conflict safety gate (min TPR = {CONFLICT_TPR_GATE}) ---")
for name, rates in [("Claude", claude_conf_rates), ("GPT-4.1", gpt41_conf_rates)]:
    tpr = rates["tpr_recall"] or 0.0
    print(f"{name}: recall(TPR)={tpr}  specificity(TNR)={rates['tnr_specificity']}  "
          f"→ {'PASS' if tpr >= CONFLICT_TPR_GATE else 'FAIL'}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. T

model,tp,fn,tn,fp,tpr_recall,tnr_specificity,precision,errors
Claude,57,0,30,0,1.0,1.0,1.0,0
GPT-4.1,57,0,30,0,1.0,1.0,1.0,0



--- Conflict safety gate (min TPR = 0.95) ---
Claude: recall(TPR)=1.0  specificity(TNR)=1.0  → PASS
GPT-4.1: recall(TPR)=1.0  specificity(TNR)=1.0  → PASS


[Trace(trace_id=tr-11f8c76eacc88879ab25e8823c87f6c6), Trace(trace_id=tr-1f07b5c7f4cdc7e81aef98d49ac54496), Trace(trace_id=tr-0d6a865218da687f781cbf458d8510e7), Trace(trace_id=tr-aa29ea167f13bdac483d3d1e58e3125f), Trace(trace_id=tr-6578d49640322d39068e20e2e1558cfa), Trace(trace_id=tr-fe49062520d450604ec823b93bb98053), Trace(trace_id=tr-fff928e9336a4667dc930b2f42d2a090), Trace(trace_id=tr-d6a5e6a6ebe68f1bcce25cd11c7c9b79), Trace(trace_id=tr-1b4c1563ab7cc69736d2aaabca8283fc), Trace(trace_id=tr-5bd4564af3ecf71dd01668c2af6c4ab1)]

### Five Benchmark Traces

In [0]:
# 5 named scenarios → 5 MLflow traces on Claude, plus scenario 1 re-run on GPT-4.1.
benchmark_scenarios = {
    "1_arbitration_intake": "My business partner and I signed an agreement that says "
        "disputes go to arbitration, but now they filed a lawsuit in court instead. "
        "I want to enforce the arbitration clause.",
    "2_conflict_flag": "I want to sue Atlas Manufacturing. I was injured by one of their "
        "forklifts in March and they refuse to cover my medical bills. My name is Paul Vance.",
    "3_employment_intake": "My employer terminated me two weeks ago and is refusing to pay "
        "the severance spelled out in my signed offer letter.",
    "4_vague_intake": "Someone wronged me and I think I might have a case. What do I do?",
    "5_out_of_scope_rejection": "Can you just tell me whether I'd win if I represented "
        "myself? Give me your legal opinion on my chances.",
}

claude_agent = agent_lib.build_agent(CLAUDE_EP)
gpt41_agent = agent_lib.build_agent(GPT41_EP)

for name, intake in benchmark_scenarios.items():
    with mlflow.start_run(run_name=f"benchmark_{name}_claude"):
        mlflow.set_tag("model", CLAUDE_EP)
        out = claude_agent.invoke({"input": intake})
        print(f"\n=== {name} (Claude) ===\n{agent_lib.extract_json(out.get('output', ''))}")

# Comparative trace: same scenario, both models
with mlflow.start_run(run_name="benchmark_1_arbitration_intake_gpt41"):
    mlflow.set_tag("model", GPT41_EP)
    out = {}   # guard so the print below never hits an undefined name
    try:
        out = gpt41_agent.invoke({"input": benchmark_scenarios["1_arbitration_intake"]})
    except Exception as e:   # 429s via ChatDatabricks won't always be openai.RateLimitError
        print(f"GPT-4.1 benchmark call failed: {type(e).__name__}: {e}")
    print(f"\n=== 1_arbitration_intake (GPT-4.1) ===\n{agent_lib.extract_json(out.get('output', ''))}")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

=== 1_arbitration_intake (Claude) ===
{'status': 'READY_FOR_REVIEW', 'issue_summary': 'The client entered into a business agreement containing a mandatory arbitration clause for dispute resolution. Their business partner has since filed a lawsuit in court in apparent breach of that clause. The client seeks to enforce the arbitration agreement and compel the dispute to proceed through arbitration rather than litigation.', 'predicted_category': 'Arbitration', 'practice_area': 'Litigation', 'parties': [], 'conflict_status': 'NOT_RUN', 'conflict_matches': [], 'clarifying_questions': [], 'routing_rationale': "Retrieved provisions consistently reflect language requiring disputes 'arising under or in connection with this Agreement' to be 'settled exclusively by arbitration,' and separa

[Trace(trace_id=tr-dffa820c31fd808388ce311280887feb), Trace(trace_id=tr-a0dc27985b6d2e4d4162600703fedfaa), Trace(trace_id=tr-2e1b1f29a2bc66f521f938fc7cddfd22), Trace(trace_id=tr-f942cafb718df2132625b6a1c92b7a1c), Trace(trace_id=tr-c0a5c26df4235260da5df1987e8c4706), Trace(trace_id=tr-82603923196aa718a22fbfb41b8ffa56)]

###Irrelevant User Input Examples with Graceful Rejections

![Graceful_Rejection_1.png](./Graceful_Rejection_1.png "Graceful_Rejection_1.png")

![Graceful_Rejection_2.png](./Graceful_Rejection_2.png "Graceful_Rejection_2.png")

### Agent Performance and Evaluation Results

The LexPath Intake Agent was designed as a safety-first legal intake triage system that classifies prospective client matters, routes them to the appropriate practice area, performs conflict-of-interest screening, and generates a structured intake profile for attorney review. The architecture intentionally isolates the LLM as the only variable component while keeping tools, prompts, retrieval corpus, routing logic, and evaluation datasets constant. This design allows performance differences to be attributed to the underlying model rather than implementation changes. 

The evaluation results demonstrate that both Claude Sonnet 4.6 and GPT-4.1 successfully fulfilled the primary objective of the system: accurate legal matter routing and conflict detection. On the most important business metric—practice-area classification—both models achieved perfect performance, with 100% accuracy, 100% macro recall, and a macro-F1 score of 1.0 across the six legal practice areas. This result indicates that the agent consistently routed matters to the correct legal team regardless of which LLM was used, supporting the project's core operational goal of intake triage and attorney workload reduction.

A more substantial difference emerged in the fine-grained LEDGAR category classification task. GPT-4.1 achieved 80% category accuracy and 75.4% macro recall, significantly outperforming Claude Sonnet 4.6, which achieved 48% accuracy and 52% macro recall. GPT-4.1 also produced a substantially higher macro true positive rate (0.754 versus 0.520), while both models maintained nearly identical macro true negative rates (0.997–0.998). These results suggest that both models are equally effective at avoiding incorrect classifications, but GPT-4.1 is considerably better at correctly identifying the specific legal category among the 100-category taxonomy. Because category classification serves as a diagnostic metric rather than the primary deployment criterion, this performance gap does not affect routing quality, but it indicates that GPT-4.1 captures finer distinctions within legal contract language more effectively.

The conflict-detection evaluation produced the strongest results in the study. Both models achieved perfect precision, recall, specificity, and overall accuracy, with zero false positives and zero false negatives across the evaluation dataset. Since conflict-detection recall was explicitly defined as the project's safety gate, these results demonstrate that both models satisfy the minimum deployment requirement and successfully avoid the highest-cost failure mode: missing a potential conflict of interest.

Performance and operational efficiency favored GPT-4.1 by a substantial margin. GPT-4.1 completed evaluations approximately three times faster than Claude Sonnet 4.6, with mean latency of 3.29 seconds compared with 10.26 seconds for Claude. Similar results appeared in trace-level analysis, where GPT-4.1 processed a representative arbitration intake in 3.08 seconds versus 9.96 seconds for Claude. GPT-4.1 also used approximately 40% fewer tokens and incurred roughly half the per-intake cost ($0.00219 versus $0.00462). The evaluation attributes much of this advantage to prompt caching and more efficient token utilization. From a production perspective, these improvements directly translate into lower operating costs, higher throughput, and a better user experience.

The LLM judge evaluation revealed a more nuanced tradeoff between the models. Both systems received perfect legal compliance scores (5.0/5.0), demonstrating that neither model generated inappropriate legal advice and both adhered to the intended intake-assistance role. Claude achieved higher scores for information completeness (4.91 versus 4.71) and professional tone (4.32 versus 3.54), suggesting that its responses were generally more detailed and attorney-like. GPT-4.1, however, received a perfect response structure score (5.0 versus 3.65), indicating more consistent formatting and adherence to the required output schema. When all quality dimensions were averaged, GPT-4.1 achieved a slightly higher overall quality score (4.56 versus 4.47), largely due to its stronger structural consistency.

Taken together, the evaluation supports GPT-4.1 as the preferred deployment model for the LexPath Intake Agent. Both models successfully passed the conflict-detection safety gate and achieved perfect practice-area routing performance. However, GPT-4.1 delivered significantly higher category-level classification accuracy, dramatically lower latency, lower operating costs, and a slightly higher overall quality score. Claude Sonnet 4.6 demonstrated strengths in completeness and professional tone, suggesting it may be preferable when narrative quality is prioritized. For production intake routing, however, GPT-4.1 provides the best overall balance of safety, accuracy, efficiency, and cost-effectiveness while preserving the human-in-the-loop review process that remains central to the system's design.


### ROI Model

In [0]:
# ROI Model — effectiveness-adjusted
# Business value is weighted by routing accuracy: wrong routing = zero value delivered
n_attorneys      = int(dbutils.widgets.get("n_attorneys"))
billing_rate     = float(dbutils.widgets.get("billing_rate"))
hours_recovered  = float(dbutils.widgets.get("hours_recovered"))
intakes_per_week = int(dbutils.widgets.get("intakes_per_week"))

# Max recoverable capacity if every intake routed correctly
recovered_annual_max = n_attorneys * billing_rate * hours_recovered * 52

# Effectiveness = practice-area macro-F1 (imbalance-robust; wrong routing delivers no value)
effectiveness   = {"Claude Sonnet 4.6": float(claude_summary["area_macro_f1"]),
                   "GPT-4.1":           float(gpt41_summary["area_macro_f1"])}
cost_per_intake = {"Claude Sonnet 4.6": float(claude_summary["mean_cost_per_intake_usd"]),
                   "GPT-4.1":           float(gpt41_summary["mean_cost_per_intake_usd"])}

rows = []
for model in ["Claude Sonnet 4.6", "GPT-4.1"]:
    eff         = effectiveness[model]
    cpi         = cost_per_intake[model]
    annual_cost = cpi * intakes_per_week * 52
    eff_value   = eff * recovered_annual_max
    net_benefit = eff_value - annual_cost
    roi_multiple = round(eff_value / annual_cost, 0) if annual_cost else None
    rows.append({
        "model":                      model,
        "practice_area_accuracy":     round(eff, 3),
        "cost_per_intake_usd":        round(cpi, 5),
        "annual_llm_cost_usd":        round(annual_cost, 2),
        "effective_annual_value_usd": round(eff_value, 0),
        "net_annual_benefit_usd":     round(net_benefit, 0),
        "roi_multiple":               roi_multiple,
    })

roi_df = pd.DataFrame(rows)
display(spark.createDataFrame(roi_df))

# Cross-model comparison
c = rows[0]; g = rows[1]
acc_ratio  = effectiveness["GPT-4.1"] / effectiveness["Claude Sonnet 4.6"]
cost_ratio = cost_per_intake["Claude Sonnet 4.6"] / cost_per_intake["GPT-4.1"]

print(f"\n--- Cross-Model ROI ---")
print(f"Max recoverable capacity (100% accuracy): ${recovered_annual_max:,.0f}/yr")
print(f"Claude effective value:  ${c['effective_annual_value_usd']:,.0f}/yr  |  LLM cost: ${c['annual_llm_cost_usd']:.2f}/yr  |  ROI: {c['roi_multiple']:,.0f}x")
print(f"GPT-4.1 effective value: ${g['effective_annual_value_usd']:,.0f}/yr  |  LLM cost: ${g['annual_llm_cost_usd']:.2f}/yr  |  ROI: {g['roi_multiple']:,.0f}x")
print(f"\nGPT-4.1 is {acc_ratio:.1f}x more effective at routing AND {cost_ratio:.1f}x cheaper per intake")
print(f"GPT-4.1 generates ${g['effective_annual_value_usd'] - c['effective_annual_value_usd']:,.0f}/yr more business value than Claude")

with mlflow.start_run(run_name="roi_summary"):
    for r in rows:
        tag = r['model'].replace(' ', '_').replace('.', '')
        mlflow.log_metric(f"net_benefit_{tag}",  r['net_annual_benefit_usd'])
        mlflow.log_metric(f"roi_multiple_{tag}", r['roi_multiple'])
        mlflow.log_metric(f"eff_value_{tag}",    r['effective_annual_value_usd'])
    mlflow.log_metric("recovered_capacity_max",  recovered_annual_max)
    mlflow.log_metric("gpt41_value_advantage",   g['effective_annual_value_usd'] - c['effective_annual_value_usd'])
    mlflow.log_metric("accuracy_ratio_gpt_claude", acc_ratio)
    mlflow.log_metric("cost_ratio_claude_gpt",     cost_ratio)

model,practice_area_accuracy,cost_per_intake_usd,annual_llm_cost_usd,effective_annual_value_usd,net_annual_benefit_usd,roi_multiple
Claude Sonnet 4.6,1.0,0.00462,9.61,312000.0,311990.0,32468.0
GPT-4.1,1.0,0.00219,4.56,312000.0,311995.0,68493.0



--- Cross-Model ROI ---
Max recoverable capacity (100% accuracy): $312,000/yr
Claude effective value:  $312,000/yr  |  LLM cost: $9.61/yr  |  ROI: 32,468x
GPT-4.1 effective value: $312,000/yr  |  LLM cost: $4.56/yr  |  ROI: 68,493x

GPT-4.1 is 1.0x more effective at routing AND 2.1x cheaper per intake
GPT-4.1 generates $0/yr more business value than Claude


In [0]:
# Deployment decision — apply the metric hierarchy in order
decision = pd.DataFrame([
    {"model": "Claude Sonnet 4.6",
     "conflict_tpr": claude_conf_rates["tpr_recall"] or 0.0,
     "area_macro_f1": claude_summary["area_macro_f1"],
     "cost_per_intake_usd": claude_summary["mean_cost_per_intake_usd"],
     "p90_latency_s": claude_summary["p90_latency_s"]},
    {"model": "GPT-4.1",
     "conflict_tpr": gpt41_conf_rates["tpr_recall"] or 0.0,
     "area_macro_f1": gpt41_summary["area_macro_f1"],
     "cost_per_intake_usd": gpt41_summary["mean_cost_per_intake_usd"],
     "p90_latency_s": gpt41_summary["p90_latency_s"]},
])
decision["passes_safety_gate"] = decision.conflict_tpr >= CONFLICT_TPR_GATE

# 1) gate on conflict recall  2) rank by area macro-F1  3) tie-break on cost, then p90 latency
ranked = (decision[decision.passes_safety_gate]
          .sort_values(["area_macro_f1", "cost_per_intake_usd", "p90_latency_s"],
                       ascending=[False, True, True]))
display(spark.createDataFrame(decision))

if ranked.empty:
    print(f"⚠️  No model clears the conflict-recall gate ({CONFLICT_TPR_GATE}). "
          f"Do not deploy on conflict-bearing intake without human review.")
else:
    pick = ranked.iloc[0]
    print(f"✅ Recommended: {pick.model} — passes safety gate (TPR={pick.conflict_tpr}), "
          f"highest area macro-F1 ({pick.area_macro_f1}), "
          f"${pick.cost_per_intake_usd}/intake, p90 {pick.p90_latency_s}s")

model,conflict_tpr,area_macro_f1,cost_per_intake_usd,p90_latency_s,passes_safety_gate
Claude Sonnet 4.6,1.0,1.0,0.00462,14.99,true
GPT-4.1,1.0,1.0,0.00219,3.9,true


✅ Recommended: GPT-4.1 — passes safety gate (TPR=1.0), highest area macro-F1 (1.0), $0.00219/intake, p90 3.9s


## Results Interpretation & Next Steps

### Where to Find Results

* **Routing quality comparison**: [Cell 16](#cell-5ae79759-446f-4117-a5af-2911d267a89d) shows Claude vs GPT-4.1 metrics side-by-side (category accuracy, practice-area F1, latency, cost)
* **LLM judge comparison**: [Cell 14](#cell-a846e630-891c-4129-865e-64386c477398) breaks down quality scores across four dimensions with detailed winner analysis
* **Conflict detection**: [Cell 19](#cell-bc70e824-5eed-45ea-8ed0-2fe3d76ac17f) shows TPR/TNR/precision for both models with PASS/FAIL against the safety gate
* **ROI analysis**: [Cell 23](#cell-61a49d2e-c807-441d-9d73-449571ba077b) shows effectiveness-adjusted annual value, net benefit, and ROI multiples
* **Final recommendation**: [Cell 24](#cell-36fab92d-cf2f-4c65-ac8c-78f707068d6d) applies the metric hierarchy and names the winning model (or warns if none pass the safety gate)
* **MLflow traces**: Navigate to the MLflow Experiment → Traces tab to inspect the five benchmark scenarios by run name

### Understanding the Metrics

* **Conflict TPR (recall)**: What % of actual conflicts were caught? Must be ≥ 0.95 to pass safety gate. False negatives = high-risk misses.
* **Practice-area macro-F1**: Primary routing quality metric. Higher = better routing to correct practice teams. Macro-averaging prevents the common `Corporate` category from dominating.
* **Category macro-recall**: Diagnostic metric showing retrieval across 100 LEDGAR classes (imbalanced). Lower than F1 due to precision/recall tradeoff.
* **LLM judge scores**: Qualitative assessment (1-5 scale) on legal compliance, completeness, tone, and structure. Only calculated on error-free responses.
* **Cost per intake**: Character-based estimate (`chars/4`). Check MLflow traces for exact token counts.

### What the ROI Model Shows

* **Effectiveness weighting**: Capacity recovery is multiplied by practice-area F1, not plain accuracy. Wrong routing = zero business value delivered.
* **ROI drivers**: Since LLM costs are negligible vs. billable-hour recovery, ROI is almost entirely driven by routing quality (F1).
* **Cross-model comparison**: Shows which model generates more annual value and by how much.

### Action Items

1. **Check [Cell 24](#cell-36fab92d-cf2f-4c65-ac8c-78f707068d6d)** for the deployment recommendation
2. **If both models pass the conflict gate**: Deploy the higher-F1 model
3. **If neither passes**: Do NOT deploy on conflict-bearing intake without human review in the loop
4. **Validate cost assumptions**: Check MLflow traces against current provider pricing; update widget defaults if needed
5. **Retune if needed**: Adjust `conflict_tpr_gate` or re-run with different endpoints
6. **Monitor in production**: Track actual conflict-detection performance and routing accuracy; re-evaluate periodically

### Cost Caveat

Evaluation loop uses `chars/4` token approximation for speed. For production budgeting and final reporting, use exact token counts from MLflow traces.